In [ ]:
!pip -q install transformers datasets accelerate scikit-learn pandas

In [ ]:
import os
import re
import zipfile
import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Dict

import torch
from sklearn.metrics import accuracy_score, f1_score, classification_report
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed
)

set_seed(42)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving test.csv to test.csv
Saving train.csv.zip to train.csv.zip
Saving validation.csv to validation.csv


In [ ]:
def load_train_from_zip(zip_path: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        assert len(names) >= 1, "Zip is empty"
        with z.open(names[0]) as f:
            return pd.read_csv(f)

train_df = load_train_from_zip("train.csv.zip")
val_df = pd.read_csv("validation.csv")
test_df = pd.read_csv("test.csv")

print(train_df.shape, val_df.shape, test_df.shape)
train_df.head()

(11118, 3) (1000, 3) (1000, 3)


,dialog,act,emotion
0,"['Say , Jim , how about going for a few beers ...",[3 4 2 2 2 3 4 1 3 4],[0 0 0 0 0 0 4 4 4 4]
1,"['Can you do push-ups ? '\n "" Of course I can ...",[2 1 2 2 1 1],[0 0 6 0 0 0]
2,"['Can you study with the radio on ? '\n ' No ,...",[2 1 2 1 1],[0 0 0 0 0]
3,['Are you all right ? '\n ' I will be all righ...,[2 1 1 1],[0 0 0 0]
4,"['Hey John , nice skates . Are they new ? '\n ...",[2 1 2 1 1 2 1 3 4],[0 0 0 0 0 6 0 6 0]


In [ ]:
display(test_df)

,dialog,act,emotion
0,"['Hey man , you wanna buy some weed ? ' ' Some...",[3 2 3 4 3 4 3 2 3 4 2 3],[0 6 0 0 0 0 0 0 0 0 3 0]
1,['The taxi drivers are on strike again . ' ' W...,[1 2 1 1],[0 0 0 0]
2,"[""We've managed to reduce our energy consumpti...",[1 2 1 2 1 2 1],[0 0 0 0 0 0 0]
3,"['Believe it or not , tea is the most popular ...",[1 1 1 1 2 2 2 2 1 1 1 3 4 3],[0 0 0 0 0 0 0 0 0 4 0 0 4 4]
4,['What are your personal weaknesses ? '\n ' I ...,[2 1 2 1 2 1 2 1],[0 0 0 0 0 0 0 4]
...,...,...,...
995,"['Frank ’ s getting married , do you believe t...",[2 2 1 2 1 2 1],[0 6 0 0 0 0 0]
996,"['OK . Come back into the classroom , class . ...",[1 2 1 1 1 2 1],[0 0 0 5 0 0 0]
997,"['Do you have any hobbies ? ' ' Yes , I like c...",[2 1 2 1 2 1 1],[0 4 4 0 6 0 0]
998,"[""Jenny , what's wrong with you ? Why do you k...",[2 1 1],[0 0 0]


In [ ]:
def parse_dialog(dialog_str: str) -> List[str]:
    """
    Extract utterances from strings like:
    ['hello' 'how are you' "fine"]
    """
    s = str(dialog_str).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1]

    matches = re.finditer(r"'([^']*)'|\"([^\"]*)\"", s)
    utterances = []
    for m in matches:
        utterances.append(m.group(1) if m.group(1) is not None else m.group(2))

    # basic cleanup
    utterances = [u.replace("\\n", " ").strip() for u in utterances if u.strip()]
    return utterances


def parse_label_list(label_str: str) -> List[int]:
    s = str(label_str).strip()
    if s.startswith("[") and s.endswith("]"):
        s = s[1:-1].strip()
    if not s:
        return []
    return [int(x) for x in s.split()]


def parse_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    bad_rows = 0

    for idx, row in df.iterrows():
        dialog = parse_dialog(row["dialog"])
        acts = parse_label_list(row["act"])
        emotions = parse_label_list(row["emotion"])

        if not (len(dialog) == len(acts) == len(emotions)):
            bad_rows += 1
            continue

        rows.append({
            "dialog_list": dialog,
            "act_list": acts,
            "emotion_list": emotions
        })

    parsed = pd.DataFrame(rows)
    print(f"Kept {len(parsed)} rows, filtered {bad_rows} malformed rows")
    return parsed


train_parsed = parse_dataframe(train_df)
val_parsed = parse_dataframe(val_df)
test_parsed = parse_dataframe(test_df)

Kept 11082 rows, filtered 36 malformed rows
Kept 997 rows, filtered 3 malformed rows
Kept 1000 rows, filtered 0 malformed rows


In [ ]:
def flatten_dialogs(df: pd.DataFrame, context_size: int = 0) -> pd.DataFrame:
    examples = []

    for _, row in df.iterrows():
        utterances = row["dialog_list"]
        acts = row["act_list"]
        emotions = row["emotion_list"]

        for i, utt in enumerate(utterances):
            if context_size > 0:
                start = max(0, i - context_size)
                context_utts = utterances[start:i]
                text = " [SEP] ".join(context_utts + [utt])
            else:
                text = utt

            examples.append({
                "text": text,
                "act_label": acts[i],
                "emotion_label": emotions[i]
            })

    return pd.DataFrame(examples)


# Start simple: no context
train_flat = flatten_dialogs(train_parsed, context_size=0)
val_flat = flatten_dialogs(val_parsed, context_size=0)
test_flat = flatten_dialogs(test_parsed, context_size=0)

print(train_flat.shape, val_flat.shape, test_flat.shape)
train_flat.head()

(86736, 3) (8019, 3) (7740, 3)


,text,act_label,emotion_label
0,"Say , Jim , how about going for a few beers af...",3,0
1,You know that is tempting but is really not go...,4,0
2,What do you mean ? It will help us to relax .,2,0
3,Do you really think so ? I don't . It will jus...,2,0
4,I guess you are right.But what shall we do ? I...,2,0


In [ ]:
TASK = "act"   # change to "emotion" for second run

label_col = "act_label" if TASK == "act" else "emotion_label"

num_labels = int(max(
    train_flat[label_col].max(),
    val_flat[label_col].max(),
    test_flat[label_col].max()
)) + 1

print("Task:", TASK)
print("Num labels:", num_labels)
print(train_flat[label_col].value_counts().sort_index())

Task: act
Num labels: 5
act_label
1    39654
2    24846
3    14187
4     8049
Name: count, dtype: int64


In [ ]:
def to_hf_dataset(df: pd.DataFrame, label_col: str) -> Dataset:
    data = df[["text", label_col]].copy()
    data = data.rename(columns={label_col: "labels"})
    return Dataset.from_pandas(data, preserve_index=False)

train_ds = to_hf_dataset(train_flat, label_col)
val_ds = to_hf_dataset(val_flat, label_col)
test_ds = to_hf_dataset(test_flat, label_col)

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize_fn(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=128
    )

train_ds = train_ds.map(tokenize_fn, batched=True)
val_ds = val_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/86736 [00:00<?, ? examples/s]

Map:   0%|          | 0/8019 [00:00<?, ? examples/s]

Map:   0%|          | 0/7740 [00:00<?, ? examples/s]

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

train_labels = train_flat[label_col].values
classes = np.arange(num_labels)

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=train_labels
)

class_weights = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", class_weights)

ValueError: classes should have valid labels that are in y

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted")
    }

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

True
Tesla T4


In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir=f"./{TASK}_bert_output",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

trainer.train()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.451500,0.515443,0.804090,0.731537,0.790394
2,0.403957,0.500977,0.813443,0.762842,0.809023
3,0.318013,0.568974,0.812820,0.760927,0.807529


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=16263, training_loss=0.3991664893980624, metrics={'train_runtime': 1292.3222, 'train_samples_per_second': 201.349, 'train_steps_per_second': 12.584, 'total_flos': 5709943254495168.0, 'train_loss': 0.3991664893980624, 'epoch': 3.0})

In [ ]:
val_metrics = trainer.evaluate(val_ds)
test_metrics = trainer.evaluate(test_ds)

print("Validation:", val_metrics)
print("Test:", test_metrics)

test_preds = trainer.predict(test_ds)
y_true = test_ds["labels"]
y_pred = np.argmax(test_preds.predictions, axis=-1)

print(classification_report(y_true, y_pred, digits=4))